# Pipeline Runner

Interactive notebook for testing and running the data pipelines locally.
Useful for debugging individual steps or running a full pipeline without deploying.

In [1]:
import sys
import os
import logging

# Add pipelines root to path so shared modules are importable
pipelines_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if pipelines_root not in sys.path:
    sys.path.insert(0, pipelines_root)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
)

import pandas as pd
pd.set_option("display.max_rows", 50, "display.max_columns", None)

from shared.season import current_season_year, is_in_season
from shared.constants import DEFAULT_SPANS, META_LABELS, STAT_LABELS

print(f"Season: {current_season_year()}")
print(f"In season: {is_in_season()}")
print(f"Default spans: {DEFAULT_SPANS}")

Season: 2026
In season: True
Default spans: [3, 5, 7]


## Configuration

Set the season, teams, and spans you want to work with.

In [2]:
SEASON = current_season_year()
IS_WOMENS = False
SPANS = DEFAULT_SPANS  # [3, 5, 7]

# Load team keys from CSV
gender = "womens" if IS_WOMENS else "mens"
teams_csv = os.path.abspath(os.path.join(pipelines_root, "..", "data", f"{gender}_teams.csv"))
teams_df = pd.read_csv(teams_csv)
ALL_TEAMS = teams_df["SR key"].tolist()

# Pick a subset for testing (set to ALL_TEAMS for full run)
TEST_TEAMS = ALL_TEAMS[:3]

print(f"Season: {SEASON}")
print(f"Gender: {'Women' if IS_WOMENS else 'Men'}")
print(f"Total teams: {len(ALL_TEAMS)}")
print(f"Test teams: {TEST_TEAMS}")
teams_df.head()

Season: 2026
Gender: Men
Total teams: 491
Test teams: ['abilene-christian', 'air-force', 'akron']


,School,"City, State",SR key,NCAA key,NCAA School,NCAA Name,background-color
0,Abilene Christian,"Abilene, Texas",abilene-christian,abilene-christian,Abilene Christian,Abilene Christian University,#582C83
1,Air Force,"USAF Academy, Colorado",air-force,air-force,Air Force,Air Force Academy,#0032A0
2,Akron,"Akron, Ohio",akron,akron,Akron,University of Akron,#0F192B
3,Alabama,"Tuscaloosa, Alabama",alabama,alabama,Alabama,University of Alabama,#9D2235
4,Alabama A&M,"Normal, Alabama",alabama-am,alabama-am,Alabama A&M,Alabama A&M University,#862633


---
## Team Stats Pipeline

Step-by-step execution of the team stats pipeline.
Each cell runs one stage so you can inspect intermediate results.

### Step 1 — Download gamelogs

Scrapes basic + advanced gamelog HTML from Sports Reference.
⚠️ This makes HTTP requests with a 3s delay per team — use `TEST_TEAMS` for quick iteration.

In [3]:
from shared.scraper import download_gamelogs

teams_to_run = TEST_TEAMS  # Change to ALL_TEAMS for full run

download_gamelogs(teams_to_run, SEASON, IS_WOMENS)
print(f"Downloaded gamelogs for {len(teams_to_run)} teams")

2026-03-03 03:05:39,828 INFO Downloading gamelogs: abilene-christian (1/3)
2026-03-03 03:05:46,912 INFO Downloading gamelogs: air-force (2/3)
2026-03-03 03:05:53,778 INFO Downloading gamelogs: akron (3/3)


Downloaded gamelogs for 3 teams


### Step 2 — Parse basic gamelogs

Converts downloaded HTML into clean CSVs.

In [4]:
from shared.parser import create_basic_gamelog
from shared.scraper import get_team_season_file_path

for key in teams_to_run:
    try:
        create_basic_gamelog(key, SEASON, IS_WOMENS)
        print(f"✓ {key}")
    except (ValueError, FileNotFoundError) as e:
        print(f"✗ {key}: {e}")

# Preview one
sample_key = teams_to_run[0]
csv_path = get_team_season_file_path(sample_key, SEASON, f"{sample_key}_basic.csv", IS_WOMENS)
pd.read_csv(csv_path).head()

✓ abilene-christian
✓ air-force
✓ akron


,Rk,Gtm,Date,Location,Opp key,Type,Rslt,Tm,Opp,OT,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,def_FG,def_FGA,def_FG%,def_3P,def_3PA,def_3P%,def_2P,def_2PA,def_2P%,def_eFG%,def_FT,def_FTA,def_FT%,def_ORB,def_DRB,def_TRB,def_AST,def_STL,def_BLK,def_TOV,def_PF
0,1.0,1.0,2025-11-03,NaN,NaN,REG (Non-Conf),W,92.0,55.0,NaN,34.0,69.0,0.493,6.0,20.0,0.300,28.0,49.0,0.571,0.536,18.0,26.0,0.692,17.0,34.0,51.0,21.0,7.0,7.0,13.0,15.0,18.0,63.0,0.286,8.0,31.0,0.258,10.0,32.0,0.313,0.349,11.0,15.0,0.733,7.0,18.0,25.0,10.0,6.0,5.0,16.0,20.0
1,2.0,2.0,2025-11-06,NaN,nebraska-omaha,REG (Non-Conf),W,73.0,71.0,NaN,25.0,57.0,0.439,7.0,21.0,0.333,18.0,36.0,0.500,0.500,16.0,21.0,0.762,9.0,18.0,27.0,15.0,8.0,3.0,14.0,23.0,21.0,45.0,0.467,6.0,19.0,0.316,15.0,26.0,0.577,0.533,23.0,31.0,0.742,3.0,16.0,19.0,10.0,11.0,3.0,15.0,20.0
2,3.0,3.0,2025-11-11,NaN,NaN,REG (Non-Conf),W,104.0,63.0,NaN,43.0,68.0,0.632,9.0,22.0,0.409,34.0,46.0,0.739,0.699,9.0,12.0,0.750,11.0,23.0,34.0,26.0,9.0,3.0,11.0,16.0,22.0,50.0,0.440,4.0,15.0,0.267,18.0,35.0,0.514,0.480,15.0,18.0,0.833,4.0,11.0,15.0,6.0,7.0,0.0,19.0,12.0
3,4.0,4.0,2025-11-14,@,stephen-f-austin,REG (Non-Conf),L,66.0,76.0,NaN,24.0,57.0,0.421,2.0,10.0,0.200,22.0,47.0,0.468,0.439,16.0,19.0,0.842,8.0,27.0,35.0,12.0,4.0,5.0,10.0,21.0,24.0,53.0,0.453,13.0,23.0,0.565,11.0,30.0,0.367,0.575,15.0,26.0,0.577,4.0,22.0,26.0,14.0,5.0,5.0,7.0,15.0
4,5.0,5.0,2025-11-18,@,texas-state,REG (Non-Conf),L,49.0,63.0,NaN,17.0,55.0,0.309,5.0,15.0,0.333,12.0,40.0,0.300,0.355,10.0,19.0,0.526,12.0,15.0,27.0,6.0,14.0,5.0,14.0,19.0,21.0,46.0,0.457,5.0,8.0,0.625,16.0,38.0,0.421,0.511,16.0,22.0,0.727,10.0,23.0,33.0,8.0,11.0,9.0,18.0,15.0


### Step 3 — Parse advanced gamelogs

In [6]:
from shared.parser import create_advanced_gamelog

for key in teams_to_run:
    try:
        create_advanced_gamelog(key, SEASON, IS_WOMENS)
        print(f"✓ {key}")
    except (ValueError, FileNotFoundError) as e:
        print(f"✗ {key}: {e}")

csv_path = get_team_season_file_path(sample_key, SEASON, f"{sample_key}_advanced.csv", IS_WOMENS)
pd.read_csv(csv_path).head()

✓ abilene-christian
✓ air-force
✓ akron


,Rk,Gtm,Date,Location,Opp key,Type,Rslt,Tm,Opp,OT,ORtg,DRtg,Pace,FTr,3PAr,TS%,TRB%,AST%,STL%,BLK%,eFG%,TOV%,ORB%,FT/FGA,def_eFG%,def_TOV%,def_ORB%,def_FT/FGA
0,1.0,1.0,2025-11-03,NaN,NaN,REG (Non-Conf),W,92.0,55.0,NaN,117.6,70.3,78.2,0.377,0.290,0.565,67.1,61.8,8.9,21.9,0.536,13.8,48.6,0.261,0.349,18.6,17.1,0.175
1,2.0,2.0,2025-11-06,NaN,nebraska-omaha,REG (Non-Conf),W,73.0,71.0,NaN,101.6,98.8,71.9,0.368,0.368,0.545,58.7,60.0,11.1,11.5,0.500,17.3,36.0,0.281,0.533,20.1,14.3,0.511
2,3.0,3.0,2025-11-11,NaN,NaN,REG (Non-Conf),W,104.0,63.0,NaN,141.3,85.6,73.6,0.176,0.324,0.706,69.4,60.5,12.2,8.6,0.699,13.0,50.0,0.132,0.480,24.5,14.8,0.300
3,4.0,4.0,2025-11-14,@,stephen-f-austin,REG (Non-Conf),L,66.0,76.0,NaN,96.8,111.5,68.2,0.333,0.175,0.500,57.4,50.0,5.9,16.7,0.439,13.2,26.7,0.281,0.575,9.7,12.9,0.283
4,5.0,5.0,2025-11-18,@,texas-state,REG (Non-Conf),L,49.0,63.0,NaN,75.1,96.6,65.2,0.345,0.273,0.383,45.0,35.3,21.5,13.2,0.355,17.9,34.3,0.182,0.511,24.2,40.0,0.348


### Step 4 — Merge basic + advanced

In [8]:
from shared.parser import combine_basic_advanced

for key in teams_to_run:
    try:
        combine_basic_advanced(key, SEASON, IS_WOMENS)
        print(f"✓ {key}")
    except FileNotFoundError as e:
        print(f"✗ {key}: {e}")

csv_path = get_team_season_file_path(sample_key, SEASON, f"{sample_key}_merged.csv", IS_WOMENS)
merged_df = pd.read_csv(csv_path)
print(f"Merged shape: {merged_df.shape}")
merged_df.head()

✓ abilene-christian
✓ air-force
✓ akron
Merged shape: (32, 68)


,Rk,Gtm,Date,Location,Opp key,Type,Rslt,Tm,Opp,OT,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,def_FG,def_FGA,def_FG%,def_3P,def_3PA,def_3P%,def_2P,def_2PA,def_2P%,def_eFG%,def_FT,def_FTA,def_FT%,def_ORB,def_DRB,def_TRB,def_AST,def_STL,def_BLK,def_TOV,def_PF,ORtg,DRtg,Pace,FTr,3PAr,TS%,TRB%,AST%,STL%,BLK%,TOV%,ORB%,FT/FGA,def_TOV%,def_ORB%,def_FT/FGA
0,1.0,1.0,2025-11-03,H,NaN,REG (Non-Conf),W,92.0,55.0,0,34.0,69.0,0.493,6.0,20.0,0.300,28.0,49.0,0.571,0.536,18.0,26.0,0.692,17.0,34.0,51.0,21.0,7.0,7.0,13.0,15.0,18.0,63.0,0.286,8.0,31.0,0.258,10.0,32.0,0.313,0.349,11.0,15.0,0.733,7.0,18.0,25.0,10.0,6.0,5.0,16.0,20.0,117.6,70.3,78.2,0.377,0.290,0.565,67.1,61.8,8.9,21.9,13.8,48.6,0.261,18.6,17.1,0.175
1,2.0,2.0,2025-11-06,H,nebraska-omaha,REG (Non-Conf),W,73.0,71.0,0,25.0,57.0,0.439,7.0,21.0,0.333,18.0,36.0,0.500,0.500,16.0,21.0,0.762,9.0,18.0,27.0,15.0,8.0,3.0,14.0,23.0,21.0,45.0,0.467,6.0,19.0,0.316,15.0,26.0,0.577,0.533,23.0,31.0,0.742,3.0,16.0,19.0,10.0,11.0,3.0,15.0,20.0,101.6,98.8,71.9,0.368,0.368,0.545,58.7,60.0,11.1,11.5,17.3,36.0,0.281,20.1,14.3,0.511
2,3.0,3.0,2025-11-11,H,NaN,REG (Non-Conf),W,104.0,63.0,0,43.0,68.0,0.632,9.0,22.0,0.409,34.0,46.0,0.739,0.699,9.0,12.0,0.750,11.0,23.0,34.0,26.0,9.0,3.0,11.0,16.0,22.0,50.0,0.440,4.0,15.0,0.267,18.0,35.0,0.514,0.480,15.0,18.0,0.833,4.0,11.0,15.0,6.0,7.0,0.0,19.0,12.0,141.3,85.6,73.6,0.176,0.324,0.706,69.4,60.5,12.2,8.6,13.0,50.0,0.132,24.5,14.8,0.300
3,4.0,4.0,2025-11-14,@,stephen-f-austin,REG (Non-Conf),L,66.0,76.0,0,24.0,57.0,0.421,2.0,10.0,0.200,22.0,47.0,0.468,0.439,16.0,19.0,0.842,8.0,27.0,35.0,12.0,4.0,5.0,10.0,21.0,24.0,53.0,0.453,13.0,23.0,0.565,11.0,30.0,0.367,0.575,15.0,26.0,0.577,4.0,22.0,26.0,14.0,5.0,5.0,7.0,15.0,96.8,111.5,68.2,0.333,0.175,0.500,57.4,50.0,5.9,16.7,13.2,26.7,0.281,9.7,12.9,0.283
4,5.0,5.0,2025-11-18,@,texas-state,REG (Non-Conf),L,49.0,63.0,0,17.0,55.0,0.309,5.0,15.0,0.333,12.0,40.0,0.300,0.355,10.0,19.0,0.526,12.0,15.0,27.0,6.0,14.0,5.0,14.0,19.0,21.0,46.0,0.457,5.0,8.0,0.625,16.0,38.0,0.421,0.511,16.0,22.0,0.727,10.0,23.0,33.0,8.0,11.0,9.0,18.0,15.0,75.1,96.6,65.2,0.345,0.273,0.383,45.0,35.3,21.5,13.2,17.9,34.3,0.182,24.2,40.0,0.348


### Step 5 — Compute moving averages

In [10]:
from shared.parser import generate_moving_averages

for span in SPANS:
    for key in teams_to_run:
        try:
            generate_moving_averages(key, SEASON, span, keep_latest=True, is_womens=IS_WOMENS)
            print(f"✓ {key} (span={span})")
        except FileNotFoundError as e:
            print(f"✗ {key} (span={span}): {e}")

# Preview one
csv_path = get_team_season_file_path(sample_key, SEASON, f"{sample_key}_5ma.csv", IS_WOMENS)
ma_df = pd.read_csv(csv_path)
print(f"MA shape: {ma_df.shape}")
ma_df.tail(3)

✓ abilene-christian (span=3)
✓ air-force (span=3)
✓ akron (span=3)
✓ abilene-christian (span=5)
✓ air-force (span=5)
✓ akron (span=5)
✓ abilene-christian (span=7)
✓ air-force (span=7)
✓ akron (span=7)
MA shape: (22, 245)


,Rk,Gtm,Date,Location,Opp key,Type,Rslt,Tm,Opp,OT,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,def_FG,def_FGA,def_FG%,def_3P,def_3PA,def_3P%,def_2P,def_2PA,def_2P%,def_eFG%,def_FT,def_FTA,def_FT%,def_ORB,def_DRB,def_TRB,def_AST,def_STL,def_BLK,def_TOV,def_PF,ORtg,DRtg,Pace,FTr,3PAr,TS%,TRB%,AST%,STL%,BLK%,TOV%,ORB%,FT/FGA,def_TOV%,def_ORB%,def_FT/FGA,OT_SMA,OT_CMA,OT_EMA,FG_SMA,FG_CMA,FG_EMA,FGA_SMA,FGA_CMA,FGA_EMA,FG%_SMA,FG%_CMA,FG%_EMA,3P_SMA,3P_CMA,3P_EMA,3PA_SMA,3PA_CMA,3PA_EMA,3P%_SMA,3P%_CMA,3P%_EMA,2P_SMA,2P_CMA,2P_EMA,2PA_SMA,2PA_CMA,2PA_EMA,2P%_SMA,2P%_CMA,2P%_EMA,eFG%_SMA,eFG%_CMA,eFG%_EMA,FT_SMA,FT_CMA,FT_EMA,FTA_SMA,FTA_CMA,FTA_EMA,FT%_SMA,FT%_CMA,FT%_EMA,ORB_SMA,ORB_CMA,ORB_EMA,DRB_SMA,DRB_CMA,DRB_EMA,TRB_SMA,TRB_CMA,TRB_EMA,AST_SMA,AST_CMA,AST_EMA,STL_SMA,STL_CMA,STL_EMA,BLK_SMA,BLK_CMA,BLK_EMA,TOV_SMA,TOV_CMA,TOV_EMA,PF_SMA,PF_CMA,PF_EMA,def_FG_SMA,def_FG_CMA,def_FG_EMA,def_FGA_SMA,def_FGA_CMA,def_FGA_EMA,def_FG%_SMA,def_FG%_CMA,def_FG%_EMA,def_3P_SMA,def_3P_CMA,def_3P_EMA,def_3PA_SMA,def_3PA_CMA,def_3PA_EMA,def_3P%_SMA,def_3P%_CMA,def_3P%_EMA,def_2P_SMA,def_2P_CMA,def_2P_EMA,def_2PA_SMA,def_2PA_CMA,def_2PA_EMA,def_2P%_SMA,def_2P%_CMA,def_2P%_EMA,def_eFG%_SMA,def_eFG%_CMA,def_eFG%_EMA,def_FT_SMA,def_FT_CMA,def_FT_EMA,def_FTA_SMA,def_FTA_CMA,def_FTA_EMA,def_FT%_SMA,def_FT%_CMA,def_FT%_EMA,def_ORB_SMA,def_ORB_CMA,def_ORB_EMA,def_DRB_SMA,def_DRB_CMA,def_DRB_EMA,def_TRB_SMA,def_TRB_CMA,def_TRB_EMA,def_AST_SMA,def_AST_CMA,def_AST_EMA,def_STL_SMA,def_STL_CMA,def_STL_EMA,def_BLK_SMA,def_BLK_CMA,def_BLK_EMA,def_TOV_SMA,def_TOV_CMA,def_TOV_EMA,def_PF_SMA,def_PF_CMA,def_PF_EMA,ORtg_SMA,ORtg_CMA,ORtg_EMA,DRtg_SMA,DRtg_CMA,DRtg_EMA,Pace_SMA,Pace_CMA,Pace_EMA,FTr_SMA,FTr_CMA,FTr_EMA,3PAr_SMA,3PAr_CMA,3PAr_EMA,TS%_SMA,TS%_CMA,TS%_EMA,TRB%_SMA,TRB%_CMA,TRB%_EMA,AST%_SMA,AST%_CMA,AST%_EMA,STL%_SMA,STL%_CMA,STL%_EMA,BLK%_SMA,BLK%_CMA,BLK%_EMA,TOV%_SMA,TOV%_CMA,TOV%_EMA,ORB%_SMA,ORB%_CMA,ORB%_EMA,FT/FGA_SMA,FT/FGA_CMA,FT/FGA_EMA,def_TOV%_SMA,def_TOV%_CMA,def_TOV%_EMA,def_ORB%_SMA,def_ORB%_CMA,def_ORB%_EMA,def_FT/FGA_SMA,def_FT/FGA_CMA,def_FT/FGA_EMA
19,29.0,29.0,2026-02-26,@,dixie-state,REG (Conf),L,81.0,85.0,0,30.0,62.0,0.484,7.0,22.0,0.318,23.0,40.0,0.575,0.540,14.0,16.0,0.875,5.0,19.0,24.0,15.0,7.0,0.0,6.0,19.0,33.0,54.0,0.611,8.0,19.0,0.421,25.0,35.0,0.714,0.685,11.0,15.0,0.733,4.0,23.0,27.0,17.0,4.0,2.0,12.0,14.0,115.9,121.7,69.9,0.258,0.355,0.582,47.1,50.0,10.0,0.0,7.9,17.9,0.226,16.4,17.4,0.204,0.2,0.041667,0.222222,23.4,23.416667,24.789198,56.6,55.458333,55.926175,0.4172,0.424958,0.446943,4.4,5.708333,4.446729,19.2,18.041667,18.133316,0.2222,0.315500,0.238765,19.0,17.708333,20.342469,37.4,37.416667,37.792859,0.5076,0.476708,0.535025,0.4576,0.477125,0.488019,19.2,15.250000,18.821705,27.0,21.791667,27.106818,0.7070,0.692125,0.679961,10.2,9.000000,9.035896,21.0,18.000000,19.078309,31.2,27.000000,28.114205,10.6,12.583333,11.574869,10.2,9.791667,11.946799,1.8,2.750000,2.328961,14.8,13.625000,14.648022,22.2,20.958333,21.553662,21.8,24.083333,23.801307,45.2,48.166667,46.355717,0.4864,0.501958,0.513023,5.0,5.875,5.628047,11.4,14.708333,12.107639,0.4216,0.394417,0.454148,16.8,18.208333,18.173260,33.8,33.458333,34.248078,0.5012,0.548458,0.531436,0.5398,0.562417,0.571940,18.4,18.791667,17.860484,26.8,25.833333,25.512728,0.6756,0.727292,0.692147,5.6,6.416667,5.397816,21.4,21.750000,21.040750,27.0,28.166667,26.438566,8.8,12.000000,9.614323,8.6,8.666667,9.681135,6.0,4.916667,6.272619,19.0,15.791667,20.344251,22.6,18.125000,22.018860,96.82,96.895833,98.436285,92.08,103.787500,96.087574,70.98,69.783333,72.032041,0.4886,0.401708,0.494394,0.3440,0.326250,0.327264,0.5102,0.517458,0.531869,53.16,48.641667,50.651043,46.12,54.195833,48.315052,13.84,13.920833,16.026505,5.28,8.333333,6.657964,17.48,17.029167,17.445630,32.20,29.266667,29.481528,0.3506,0.281833,0.346053,24.72,20.658333,25.765133,20.68,26.579167,22.691866,0.4126,0.396792,0.394905
20,30.0,30.0,2026-02-28,@,utah-valley,REG (Conf),L,67.0,74.0,0,18.0

### Step 6 — Run the full team stats pipeline

This runs all steps end-to-end and produces the JSON output.
Without `AZURE_STORAGE_CONNECTION_STRING`, results are written locally.

In [ ]:
from shared.stats import generate_and_upload_team_stats

# Run the full pipeline (writes local JSON if no connection string)
generate_and_upload_team_stats(SEASON)
print("Done — check for ncaam_basketball_team_stats.json / ncaaw_basketball_team_stats.json")

---
## Top 25 Pipeline

Generates AP Top 25 power rankings by running pairwise predictions through the API.

⚠️ Requires the API to be running (either locally or the deployed endpoint).

In [ ]:
from shared.scraper import get_ap_top_25

top25_men = get_ap_top_25(SEASON, is_womens=False)
print(f"Men's AP Top 25: {len(top25_men)} teams")
for i, team in enumerate(top25_men, 1):
    print(f"  #{i} {team}")

In [ ]:
from top25.generate import generate_and_upload_top25

# Use deployed API by default; change to http://localhost:8000 for local
API_URL = "https://mlmb-api.purplesand-9a1718e2.eastus.azurecontainerapps.io"

generate_and_upload_top25(API_URL, SEASON)
print("Done — check for ncaam_basketball_top25.json / ncaaw_basketball_top25.json")

---
## Utilities

Helpers for inspecting data at any point.

In [ ]:
# Inspect any team's files for the current season
def list_team_files(school_key: str, season: int = SEASON, is_womens: bool = IS_WOMENS):
    """List all generated files for a team/season."""
    from shared.scraper import get_data_dir
    data_dir = get_data_dir(school_key, season, is_womens)
    if os.path.exists(data_dir):
        files = os.listdir(data_dir)
        print(f"{data_dir}:")
        for f in sorted(files):
            size = os.path.getsize(os.path.join(data_dir, f))
            print(f"  {f} ({size:,} bytes)")
    else:
        print(f"No data directory for {school_key} ({season})")

list_team_files(ALL_TEAMS[0])

In [ ]:
# Quick preview of any team CSV
def preview_team_csv(school_key: str, suffix: str = "merged", season: int = SEASON, is_womens: bool = IS_WOMENS):
    """Load and display a team CSV. suffix: basic, advanced, merged, 5ma, 5span_full, etc."""
    csv_path = get_team_season_file_path(school_key, season, f"{school_key}_{suffix}.csv", is_womens)
    df = pd.read_csv(csv_path)
    print(f"{csv_path}")
    print(f"Shape: {df.shape}")
    return df

preview_team_csv(ALL_TEAMS[0], "merged")